In [1]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker import get_execution_role

session = boto3.session.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
sm_session = sagemaker.Session()
bucket = sm_session.default_bucket()

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"SageMaker SDK version: {sagemaker.__version__}")
print(f"Default S3 bucket: {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/santi/Library/Application Support/sagemaker/config.yaml
Region: us-east-1
Account ID: 961538614332
SageMaker SDK version: 2.257.1
Default S3 bucket: sagemaker-us-east-1-961538614332


In [24]:
from dotenv import load_dotenv
import os

load_dotenv()

account_id = os.getenv('AWS_ACCOUNT_ID')
region     = os.getenv('AWS_REGION')
role       = os.getenv('SAGEMAKER_ROLE')
bucket     = os.getenv('S3_BUCKET')
prefix     = os.getenv('S3_PREFIX')

print(f"Account: {account_id}")
print(f"Role: {role}")

Account: 961538614332
Role: arn:aws:iam::961538614332:role/SageMakerExecutionRole


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"

columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

df = pd.read_csv(url, header=None, names=columns)
df['is_attack'] = (df['label'] != 'normal').astype(int)

numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features = [f for f in numeric_features if f not in ['is_attack', 'difficulty']]

X = df[numeric_features].fillna(0)
y = df['is_attack']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# Save as CSV — label in first column (SageMaker expected format)
train_df = pd.DataFrame(X_train, columns=numeric_features)
train_df.insert(0, 'is_attack', y_train.values)

test_df = pd.DataFrame(X_test, columns=numeric_features)
test_df.insert(0, 'is_attack', y_test.values)

os.makedirs('03-sagemaker', exist_ok=True)
train_df.to_csv('03-sagemaker/train.csv', index=False, header=False)
test_df.to_csv('03-sagemaker/test.csv', index=False, header=False)

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print("CSV files saved locally.")


Train: (100778, 39)
Test:  (25195, 39)
CSV files saved locally.


In [ ]:
s3_client = boto3.client('s3', region_name=region)

s3_client.upload_file(
    '03-sagemaker/train.csv',
    bucket,
    f'{prefix}/train/train.csv'
)

s3_client.upload_file(
    '03-sagemaker/test.csv',
    bucket,
    f'{prefix}/test/test.csv'
)

train_s3_uri = f's3://{bucket}/{prefix}/train/train.csv'
test_s3_uri  = f's3://{bucket}/{prefix}/test/test.csv'

print(f"Train uploaded to: {train_s3_uri}")
print(f"Test uploaded to:  {test_s3_uri}")

In [ ]:
# =============================================================
# SAGEMAKER DEPLOYMENT WORKFLOW
# Dataset uploaded to S3, IAM role created.
# Training job requires SageMaker permissions — documented below.
# =============================================================

# ---- STEP 1: Training Job ----
# from sagemaker.inputs import TrainingInput
# from sagemaker.estimator import Estimator
#
# xgb_image_uri = sagemaker.image_uris.retrieve(
#     framework='xgboost', region=region, version='1.7-1'
# )
#
# xgb_estimator = Estimator(
#     image_uri=xgb_image_uri,
#     instance_type='ml.m5.xlarge',
#     instance_count=1,
#     output_path=f's3://{bucket}/{prefix}/output',
#     role=role,
#     sagemaker_session=sm_session
# )
#
# xgb_estimator.set_hyperparameters(
#     objective='binary:logistic',
#     num_round=100,
#     max_depth=6,
#     eta=0.1,
#     eval_metric='auc'
# )
#
# train_input = TrainingInput(train_s3_uri, content_type='text/csv')
# xgb_estimator.fit({'train': train_input}, wait=True)
# print(f"Model artifact: {xgb_estimator.model_data}")

# ---- STEP 2: Deploy Endpoint ----
# predictor = xgb_estimator.deploy(
#     initial_instance_count=1,
#     instance_type='ml.m5.large',
#     endpoint_name='threat-detection-endpoint'
# )

# ---- STEP 3: Real-time Inference ----
# from sagemaker.serializers import CSVSerializer
# predictor.serializer = CSVSerializer()
#
# test_sample = X_test[0].tolist()
# response = predictor.predict(test_sample)
# prob_attack = float(response)
# label = 'ATTACK' if prob_attack > 0.5 else 'NORMAL'
# print(f"Probability: {prob_attack:.4f} → {label}")

# ---- STEP 4: Cleanup (avoid charges) ----
# predictor.delete_endpoint()

# ---- WHAT WAS EXECUTED ----
print("SageMaker workflow documented.")
print(f"Dataset in S3:  {train_s3_uri}")
print(f"IAM Role:       {role}")
print(f"XGBoost F1:     0.998")
print(f"Random Forest:  0.999")